In [40]:
from pathlib import Path

class CONLLUParser:
    def __init__(self):
        self.sentences = []
        self.texts = []

    def __make_wordform(self, parts):
        return {
            "id": int(parts[0]),
            "form": parts[1],
            "lemma": parts[2],
            "upos": parts[3],
            "head": int(parts[6]) if parts[6] != "_" else 0,
            "deprel": parts[7]
        }

    def __read_sentence(self, lines, pos):
        sent = []
        text = ""

        while pos < len(lines):
            line = lines[pos].rstrip("\n")

            if line == "":
                pos += 1
                break

            if line.startswith("# text = "):
                text = line[9:].strip()
            elif not line.startswith("#"):
                parts = line.split("\t")
                if len(parts) >= 8 and "-" not in parts[0] and "." not in parts[0]:
                    try:
                        sent.append(self.__make_wordform(parts))
                    except:
                        pass

            pos += 1

        return sent, text, pos

    def parse_conllu_file(self, filename):
        with open(filename, "r", encoding="utf-8") as f:
            lines = f.readlines()

        self.sentences = []
        self.texts = []

        pos = 0
        while pos < len(lines):
            sent, text, pos = self.__read_sentence(lines, pos)
            if sent:
                self.sentences.append(sent)
                self.texts.append(text)

    def find_construction(self, child_upos, child_deprel, parent_upos):
        results = []

        for sent_idx, sent in enumerate(self.sentences):
            id2word = {w["id"]: w for w in sent}

            for child in sent:
                if child["upos"] == child_upos and child["deprel"] == child_deprel:
                    parent = id2word.get(child["head"])
                    if parent and parent["upos"] == parent_upos:
                        results.append({
                            "text": self.texts[sent_idx],
                            "sentence": [w["form"] for w in sent],
                            "match": child,
                            "parent": parent
                        })
                        break

        return results


class ConstructionFinder:
    def __init__(self):
        self.parser = CONLLUParser()
        self.results = {}

    def find_in_directory(self, path, child_upos, child_deprel, parent_upos, filename_mask="*Gaydar*.conllu"):
        all_results = {}

        for file_path in Path(path).glob(filename_mask):
            self.parser.parse_conllu_file(str(file_path))
            found = self.parser.find_construction(child_upos, child_deprel, parent_upos)
            all_results[str(file_path)] = found
            self.results[str(file_path)] = found

        return all_results

In [41]:
import numpy as np

def parse_feature_name(feat_name):
    feat_name = feat_name.strip().strip("'").strip('"').strip("()")
    parts = [p.strip() for p in feat_name.split(",")]
    if len(parts) != 4:
        return None

    left, arrow, right, deprel = parts

    if arrow == "->":
        parent_upos = left
        child_upos = right
    elif arrow == "<-":
        parent_upos = right
        child_upos = left
    else:
        return None

    return parent_upos, child_upos, deprel


def feature_exists_in_author_text(author_name, feat_name, finder, conllu_dir="./conllu_DeepPavlov"):
    parsed = parse_feature_name(feat_name)
    if parsed is None:
        return False

    parent_upos, child_upos, deprel = parsed

    results = finder.find_in_directory(
        conllu_dir,
        child_upos=child_upos,
        child_deprel=deprel,
        parent_upos=parent_upos,
        filename_mask=f"*{author_name}*.conllu"
    )

    return any(len(items) > 0 for items in results.values())


def get_n_features_sorted(df, selec_type, n, finder, feature_names, author_id_to_name,
                          conllu_dir="./conllu_DeepPavlov"):
    feature_names_np = np.array(feature_names)

    for idx, row in df.iterrows():
        features = list(row["features"])
        selec_type_row = list(row[selec_type])
        author_id = row["author_id"]
        stats = list(row["feature_stats"])
        author_name = author_id_to_name[author_id]

        indexed_ = [(value, i) for i, value in enumerate(selec_type_row)]
        sorted_indexed_ = sorted(indexed_, key=lambda item: item[0], reverse=True)

        filtered_imp = []
        filtered_features = []
        filtered_stats = []
        filtered_feature_names = []

        for imp, i in sorted_indexed_:
            feat_idx = features[i]
            feat_name = feature_names_np[feat_idx]

            if feature_exists_in_author_text(author_name, feat_name, finder, conllu_dir):
                filtered_imp.append(imp)
                filtered_features.append(feat_idx)
                filtered_stats.append(stats[i])
                filtered_feature_names.append(feat_name)

            if len(filtered_imp) == n:
                break

        df.at[idx, "author"] = author_name
        df.at[idx, selec_type] = filtered_imp
        df.at[idx, "feature_stats"] = filtered_stats
        df.at[idx, "features"] = filtered_features
        df.at[idx, "feature_names"] = filtered_feature_names

    return df

In [78]:
patterns = [
    # (VERB, ->, ADV, conj)
    ("VERB", "->", "ADV", "conj", "VERB -> ADV, conj"),
    
    # (NOUN, <-, PRON, det)
    ("NOUN", "<-", "PRON", "det", "NOUN <- PRON, det"),
    
    # (NOUN, ->, PROPN, acl)
    ("NOUN", "->", "PROPN", "acl", "NOUN -> PROPN, acl"),
    
    # (NOUN, ->, CCONJ, cc)
    ("NOUN", "->", "CCONJ", "cc", "NOUN -> CCONJ, cc"),
    
    # (VERB, ->, VERB, ccomp)
    ("VERB", "->", "VERB", "ccomp", "VERB -> VERB, ccomp"),
    
    # (NOUN, ->, PROPN, acl) — дубликат
    ("NOUN", "->", "PROPN", "acl", "NOUN -> PROPN, acl"),
    
    # (VERB, ->, DET, nsubj)
    ("VERB", "->", "DET", "nsubj", "VERB -> DET, nsubj"),
    
    # (VERB, ->, VERB, xcomp)
    ("VERB", "->", "VERB", "xcomp", "VERB -> VERB, xcomp"),
    
    # (ADJ, ->, VERB, ccomp)
    ("ADJ", "->", "VERB", "ccomp", "ADJ -> VERB, ccomp"),
    
    # (PAD, <-, DET, det)
    ("PAD", "<-", "DET", "det", "PAD <- DET, det"),
    
    # (NOUN, ->, NOUN, nsubj)
    ("NOUN", "->", "NOUN", "nsubj", "NOUN -> NOUN, nsubj"),
    
    # (NOUN, ->, AUX, acl:relcl)
    ("NOUN", "->", "AUX", "acl:relcl", "NOUN -> AUX, acl:relcl"),
    
    # (VERB, ->, VERB, xcomp) — дубликат
    ("VERB", "->", "VERB", "xcomp", "VERB -> VERB, xcomp"),
    
    # (VERB, ->, PAD, xcomp)
    ("VERB", "->", "PAD", "xcomp", "VERB -> PAD, xcomp"),
    
    # (NOUN, ->, PROPN, acl) — дубликат
    ("NOUN", "->", "PROPN", "acl", "NOUN -> PROPN, acl"),
]

In [79]:
from typing import Dict, List, Tuple

# Объявите ваш finder один раз
finder = ConstructionFinder()

filename_masks = ["*Koni*.conllu", "*.conllu"]

for parent_upos, direction, child_upos, deprel, label in patterns:
    print(f"\n{label}")
    
    example_found = False
    
    # Попробовать сначала Gaydar, потом все файлы
    for mask in filename_masks:
        # Интерпретируем направление стрелки
        if direction == "<-":
            parent, child = child_upos, parent_upos
        else:  # "->"
            parent, child = parent_upos, child_upos

        # Пускаем поиск
        results = finder.find_in_directory(
            "./conllu_DeepPavlov",
            child_upos=child,
            child_deprel=deprel,
            parent_upos=parent,
            filename_mask=mask,
        )
        amount = [len(items) for items in results.values()]
        amount = sum(amount)
        print(amount)
        # Берем первый файл и первый пример
        for filename, items in results.items():
            if not items:
                continue

            item = items[0]
            
            print(f'"{item["text"]}"')
            print("MATCH:", item["match"])
            print("PARENT:", item["parent"])
            print(f"Файл: {filename}")
            print("-" * 50)
            
            example_found = True
            break  # один пример на признак

        if example_found:
            break  # выходим из маски, уже нашли пример

    if not example_found:
        print("Не найдено ни одного примера для этого признака.")


VERB -> ADV, conj
32
"Напротив, теперь-то ей и любить его, когда он всецело ей принадлежит, когда ей не надо нарушать «их закон», а между тем она обвиняет его, повторяет это обвинение здесь, на суде."
MATCH: {'id': 18, 'form': 'надо', 'lemma': 'надо', 'upos': 'ADV', 'head': 13, 'deprel': 'conj'}
PARENT: {'id': 13, 'form': 'принадлежит', 'lemma': 'принадлежать', 'upos': 'VERB', 'head': 6, 'deprel': 'advcl'}
Файл: conllu_DeepPavlov\Koni_Court speech.txt.conllu
--------------------------------------------------

NOUN <- PRON, det
0
24
"Что т-такого опасного было в его поездке?"
MATCH: {'id': 2, 'form': 'т-такого', 'lemma': 'т', 'upos': 'NOUN', 'head': 1, 'deprel': 'det'}
PARENT: {'id': 1, 'form': 'Что', 'lemma': 'что', 'upos': 'PRON', 'head': 4, 'deprel': 'nsubj'}
Файл: conllu_DeepPavlov\Akunin_Smert Ahillesa.txt.conllu
--------------------------------------------------

NOUN -> PROPN, acl
1
"Бумажки эти оказались той же фабрикации, и между ними являются пробелы, именно они идут через но

In [24]:
from copy import copy
from collections import defaultdict, Counter
import pathlib
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression, \
                                 LinearRegression
from sklearn.metrics import precision_recall_fscore_support

from pprint import pprint

# import conllu
from scipy.sparse import csr_matrix
import pymorphy3

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from umap.umap_ import UMAP

In [25]:
morph = pymorphy3.MorphAnalyzer()
morpho_cashe = defaultdict(str)

# Расчёт статистики всех связей в фрагменте книги с учётом их направления.
def calc_syntax_stat(book):
    con_stat = defaultdict(int)
    overall = 0
    for sentence in book:
        overall += len(sentence)
        for child_position, word in enumerate(sentence):
            if word['head'] is None:
                continue
            parent_position = word['head'] - 1
            if parent_position == -1:
                continue
            if parent_position > child_position:
                d = '<-'
            else:
                d = '->'
            con_stat[f"({sentence[parent_position]['upos']}, {d}, {word['upos']}, {word['deprel']})"] += 1

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

def calc_homonymy_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            if morpho_cashe[word['form']] != '':
                con_stat[morpho_cashe[word['form']]] += 1
            else:
                res = morph.parse(word['form'])
                nfs = list(set(a.normal_form for a in res))
                poses = list(set(a.tag.POS for a in res))

                if len(nfs) > 1: # Ambigous by lemma
                    if len(poses) > 1:
                        con_stat['pos_lemma'] += 1
                        morpho_cashe[word['form']] = 'pos_lemma'
                    else:
                        con_stat['lemma'] += 1
                        morpho_cashe[word['form']] = 'lemma'
                else:
                    if len(poses) > 1:
                        con_stat['pos'] += 1
                        morpho_cashe[word['form']] = 'pos'
                    else:
                        if len(res) > 1:
                            con_stat['feat'] += 1
                            morpho_cashe[word['form']] = 'feat'
                        else:
                            con_stat['unambig'] += 1
                            morpho_cashe[word['form']] = 'unambig'


    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

# service_poses = {'ADVB', 'NPRO', 'PRED', 'PREP', 'CONJ', 'PRCL', 'INTJ'}
service_poses = {'ADV', 'PRON', 'ADP', 'CCONJ', 'SCONJ', 'PART', 'DET'}

def calc_service_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            if word['upos'] in service_poses:
                con_stat[f'{word["lemma"]}_{word["upos"]}'] +=1

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

def calc_char_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            res = Counter(word['form'])
            for c, f in res.items():
                con_stat[c] += f

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

In [26]:
class CONLLUParser:
    '''
    Собственный класс для разбора CONLLU файлов.
    Потому что стандартный какой-то очень медленный.
    '''
    sentences: list # Список предложения текста, содержащих синтаксическую информацию.
    texts: list # Список предложений в текстовой форме.

    def __init__(self):
        self.sentences = []
        self.texts = []

    # Превращает кортеж в словарь слова.
    def __make_wordform(self, word):
        res = dict()
        res['id'] = int(word[0])
        res['form'] = word[1]
        res['lemma'] = word[2]
        res['upos'] = word[3]
        res['head'] = int(word[6])
        res['deprel'] = word[7]
        return res

    def __read_sentence(self, lines, pos):
        ''' Возвращает предложение из списка строк lines, записанных в формате UD.
            Предложение находится начиная со строки с номером pos.
        '''
        sent = []
        text = ""
        while pos < len(lines) and lines[pos] != '' and lines[pos] != '\n':
            try:
              if lines[pos][0] != '#':
                  sent.append(self.__make_wordform(lines[pos][:-2].split("\t")))
              elif lines[pos].startswith('# text = '):
                  text = lines[pos][9:]
            except:
              print(f"Exception: {lines[pos]}")
            pos += 1
        pos += 1
        return sent, text, pos

    # Считывает книгу в conllu из списка строк.
    def parse_conllu_lines(self, lines):
        del self.sentences
        del self.texts
        self.sentences = []
        self.texts = []
        pos = 0
        while pos < len(lines):
            sent, text, pos = self.__read_sentence(lines, pos)
            self.sentences.append(sent)
            self.texts.append(text)

    # Считывает книгу в conllu из файла.
    def parse_conllu_file(self, filename):
        with open(filename, "rt", encoding='utf-8') as book_file:
            lines = book_file.readlines()
            self.parse_conllu_lines(lines)
            del lines

In [27]:
class WriterStorage:
    '''
        Класс служит для расчёта показателей, по коорым будет определяться авторство.
      Из модуля calc_stat_vectors выгружаются функции, которые считают параметры:
      частоты по синтаксису, грамматической неоднозначности, символам, служебным частям речи.
      Он умеет сеарилизовать себя в json, потому что когда считаешь фрагментами по 10 предложений,
      по большой коллекции тебе мало 64Г оперативки.
    '''

    # Описание свойств
    book_authors: list # Список имён авторов по книгам.
    book_names: list # Список названий книг.
    authors_id: dict # Словарь соответствий фамилии автора и его идентификатора.
    book_author_id: list # Список идентификаторов авторов по книгам.
    parts_author_id: list # Список идентификаторов авторов по частям.
    book_no: list # Номер книги в списке.

    books_stat: list # Статистика по книгам в виде словаря.
    parts_stat: list # Статистика по частям в виде словаря.
    part_no: list # Номер части в книге.
    part_book_no: list # Номер книги соответствующей части.
    books_vect_dict: dict # Словарь признаков и их номеров в векторе по книгам в целом.
    books_vect: list # Векторы признаков по книгам в целом.
    parts_vect_dict: dict # Словарь признаков и их номеров в векторе по всем частям книг.
    parts_vect: list # Векторы признаков по частям книг.
    texts_by_parts: list # Тексты по частям.

    def __init__(self):
        self.book_names = []
        self.book_authors = []
        self.book_sentences = []
        self.authors_id = dict()

    # Словарь с функциями, коорые используются для расчёта. Ключ передается в функцию в качестве параметра.
    __func_correspondence  = {'syntax': calc_syntax_stat,
                              'service': calc_service_stat,
                              'homonymy': calc_homonymy_stat,
                              'char': calc_char_stat
                             }

    def calc_connections_distr(self, book, text, what='syntax', fragment_len = 100):
        '''
           Функция расчёта параметров для одной книги.
           book: Текст книги с синтаксической разметкой, разделенный на предложения.
           text: Текст книги, разделенный на предложения. Нужен, чтобы можно было посмореть где что.
           what: Какие параметры надо считать. Определяет какая функция расчёта параметров будет вызвана.
           fragment_len: Длина фрагментов, на кототрые надо делить.
           Возвращает синтаксически рамеченные предложения и текст книги, разделенные фрагменты.
           Если fragment_len = -1, возвращает книгу целиком.
        '''
        # Проверка, есть ли у нас такой ключ.
        if what not in WriterStorage.__func_correspondence.keys():
            raise Exception(f'We calculate statistics for the followig keys: '
                            f'{", ".join(WriterStorage.__func_correspondence.keys())}')

        # Расчёт статистики.
        # Определяем по какой функции ведётся расчёт.
        calc_connections_stat = WriterStorage.__func_correspondence[what]

        # -1 означает,что стаисика считаеттся для книги в целом.
        if fragment_len == -1:
            distr = calc_connections_stat(book)
            texts = [[text]]
        # В проивном случае надо разделить книгу на фрагменты.
        else:
            distr = []
            texts = []
            for i in range(0, len(book), fragment_len):
                # print(i, fragment_len)
                # print(book[i: i+fragment_len])
                distr.append(calc_connections_stat(book[i: i+fragment_len]))
                texts.append(text[i: i+fragment_len])
        return distr, texts

    def make_vector(self, data, keys):
        ''' Возвращает данные, собранные в виде словаря data в виде вектора,
            Соблюдая при этом порядок следования параметров, задаваемый при помощи keys.
        '''
        vect = np.zeros((len(keys)))
        for key, val in data.items():
            vect[keys[key]] = val
        return vect

    def vectorize_data(self, data):
        '''
        Векторизует данные из словаря data.
        Возвращает словарь признаков с номерами их позиций в векторе и сам вектор.
        '''
        data_dict = []
        for part in data:
            data_dict.extend(part.keys()) # data_dict |= set(part.keys())
        data_dict = list(set(data_dict))
        data_dict = {key:i for i, key in enumerate(data_dict)}

        # Теперь создаем разреженный массив с результирующими векторами.
        res_list = []
        rows = []
        cols = []
        for row_no, part in enumerate(data):
            res_list.extend(part.values())
            cols.extend([data_dict[key] for key in part.keys()])
            rows.extend([row_no for key in part.keys()])

        ret_data = csr_matrix((res_list, (rows, cols)), (len(data), len(data_dict))).toarray()

        # vects_a = []
        # for part in data:
        #     vects_a.append(self.make_vector(part, data_dict))
        return data_dict, ret_data

    def process_books(self, path, what='syntax', fragment_size=100):
        '''
        Функция обработки книг из каталога path по признакам, задаваемым параметром what.
        Режет текст на фрагменты размера fragment_size, если он равен -1, возвращает данные для книг целиком.
        '''
        parser = CONLLUParser() # Будет разбирать CONLLU.

        files2 = pathlib.Path(path).glob('*.conllu')
        self.books_stat = []
        self.parts_stat = []
        self.book_no = []
        self.part_no = []
        self.book_author_id = []
        self.parts_author_id = []
        self.authors_id = dict()
        self.part_book_no = []
        self.texts_by_parts = []
        self.fragment_size = fragment_size

        # Перебираем книги в CONLLU-формате.
        for i, file in enumerate(list(files2)):
            # Информация по книге. Выделяем из имени файла автора и название книги.
            # print(i, file.name)
            a = file.name.split("_")
            if a[0][-1] >= '0' and a[0][-1] <= '9':
                a[0] = a[0][:-1]
            self.book_authors.append(a[0])
            self.book_names.append(a[1][:-11])
            if a[0] not in self.authors_id.keys():
                auth_no = len(self.authors_id.keys())
                self.authors_id[a[0]] = auth_no
            else:
                auth_no = self.authors_id[a[0]]
            self.book_author_id.append(auth_no)
            self.book_no.append(i)
            print(i, a[0], a[1][:-11])

            # Читаем CONLLU.
            parser.parse_conllu_file(file.absolute())
            # book, text = parser.sentences, parser.texts
            # self.book_sentences.append(book)
            if fragment_size==-1:
                al, all_texts = self.calc_connections_distr(parser.sentences,  parser.texts, what, -1)
                self.books_stat.append(al)
            else:
                parts, parts_texts = self.calc_connections_distr(parser.sentences, parser.texts, what, fragment_size)
                # добавляем информацию и статисику по фрагментам.
                self.parts_stat.extend(parts)
                self.parts_author_id.extend([auth_no for part in parts])
                self.part_no.extend([p_n for p_n in range(len(parts))])
                self.part_book_no.extend([i for p_n in range(len(parts))])
                self.texts_by_parts.extend([t for t in parts_texts])

        # Векторизуем книги и фрагменты.
        if fragment_size == -1:
            d, v = self.vectorize_data(self.books_stat)
            self.books_vect_dict = d
            self.books_vect = v
        else:
            d, v = self.vectorize_data(self.parts_stat)
            self.parts_vect_dict = d
            self.parts_vect = v
        self.back_id = {i: a for a, i in self.authors_id.items()}

In [28]:
writers = WriterStorage()

writers.process_books('conllu_DeepPavlov', 'syntax', -1)

0 Abgaryan dalshe zhit
1 Abgaryan molchaie
2 Abgaryan 3apples
3 Abgaryan Fantastychysi
4 Abgaryan Jubileum
5 Abgaryan Ponaekhavshaya
6 Abgaryan Simon
7 Adamov Izgnanine vladyki
8 Adamov Pobediteli nedr
9 Adamov Tayna dvuh okeanov
10 Aksakov Vol1
11 Aksakov Vol2
12 Aksakov Vol3
13 Aksakov Vol4
14 Aksakov Vol5
15 AkuninBorisova Kreativschik
16 AkuninBrusnikin Bellona
17 AkuninBrusnikin Devyatny spas
18 AkuninBrusnikin Hero of another time
19 Akunin Almaznaya kolesnitsa 1
20 Akunin Almaznaya kolesnitsa 2
21 Akunin Azazel
22 Akunin Grom Pobedy
23 Akunin Leviafan
24 Akunin Lubovnik smerti
25 Akunin Nefritovye chetki
26 Akunin No West and East
27 Akunin Smert Ahillesa
28 Akunin Turetski gambit
29 Akunin World is theatre
30 Aldanov Armageddon
31 Aldanov Begstvo
32 Aldanov Bred
33 Aldanov Erfurtskoe svidanie
34 Aldanov Istoricheskie portrety
35 Aldanov Kartiny Oktyabrskogo perevorota
36 Aldanov Kluch
37 Aldanov Mogila voina
38 Aldanov Nachalo konca
39 Aldanov Ogon i dym
40 Aldanov Peschera
41 

In [32]:
author_id_to_name = {v: k for k, v in writers.authors_id.items()}
for i, item in enumerate(writers.parts_stat):
  for feat, score in item.items():
    if 'PRON <- SCONJ, cc' in feat:
      print(writers.texts_by_parts[i])
      author = author_id_to_name[writers.parts_author_id[i]]
      print(author, score)

In [34]:
books_features = np.array(writers.books_vect)
book_author_ids = np.array(writers.book_author_id)
author_ids = np.array(list(writers.authors_id.values()))
author_names = np.array(list(writers.authors_id.keys()))
author_id_to_name = {v: k for k, v in writers.authors_id.items()}

author_feature_means = []
def normalize_array(data):
    for i, array in enumerate(data):
        sum = np.sum(array)
        data[i] = array / sum
    return data

for author_id in author_ids:
    author_mask = (book_author_ids == author_id)
    author_feature_means.append(np.mean(books_features[author_mask], axis=0))

feature_authors = np.array(author_feature_means)
feature_authors = normalize_array(feature_authors)
counter_features = Counter()

for author in feature_authors:
    for i, feature in enumerate(author):
        if feature > 0:
          counter_features[i] += 1
        else:
          counter_features[i] += 0

print(counter_features[1270])

76


In [37]:
feature_authors = np.array(author_feature_means)

feature_authors[0][1270]

6.3309170333322776e-06